# NanoGPT Training!

This notebook trains a `nanoGPT` model at roughly 54 million parameters for 2 epochs. Insofar it is my best performing one yet, but takes arouned ~1.5 times longer to train this than `miniGPT`.

In [ ]:
import gc
import json
import os
import sys
from pathlib import Path

import torch
from datasets import load_dataset

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

from nanoGPT import GPTConfig, nanoGPT, tokenize_texts_to_blocks # make sure you import correctly

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
raw = load_dataset("corbt/all-recipes")

if "train" in raw:
    base_split = raw["train"]
else:
    first_key = list(raw.keys())[0]
    base_split = raw[first_key]

split = base_split.train_test_split(test_size=0.2, seed=42)
train_full = split["train"]
test_set = split["test"]

candidate_columns = ["text", "recipe", "instructions", "directions", "title", "ingredients", "source", "input"]
available_columns = train_full.column_names
text_column = next((column for column in candidate_columns if column in available_columns), None)
if text_column is None and len(available_columns) == 1:
    text_column = available_columns[0]
if text_column is None:
    raise ValueError(f"No expected text column found. Available: {available_columns}")

subset_size = min(400000, len(train_full))
train_subset = train_full.shuffle(seed=42).select(range(subset_size))


def flatten_to_text(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        return " ".join(flatten_to_text(item) for item in value)
    if isinstance(value, dict):
        for key in ["text", "input", "recipe", "instructions", "directions", "title", "ingredients"]:
            if key in value:
                return flatten_to_text(value[key])
        return " ".join(f"{key}: {flatten_to_text(item)}" for key, item in value.items())
    return str(value)

train_texts = [flatten_to_text(row.get(text_column)).strip() for row in train_subset]
train_texts = [text for text in train_texts if text]

print(f"Selected text column: {text_column}")
print(f"train_subset used: {len(train_texts):,} samples")
print(f"test_set available: {len(test_set):,} samples")

In [ ]:
config = GPTConfig(
    block_size=512,
    n_layers=9,
    n_embd=512,
    n_heads=8,
    dropout=0.1,
    use_rope=False,
    gradient_checkpointing=True,
    compile_model=True,
)

trainer = nanoGPT.from_tokenizer_name("gpt2", config=config)
trainer = trainer.to(DEVICE)

param_count = sum(parameter.numel() for parameter in trainer.model.parameters())
trainable_count = sum(parameter.numel() for parameter in trainer.model.parameters() if parameter.requires_grad)
print(f"Total parameters: {param_count:,}")
print(f"Trainable parameters: {trainable_count:,}")

In [ ]:
cache_dir = NOTEBOOK_DIR / "cache"
cache_dir.mkdir(exist_ok=True)
block_cache = cache_dir / "train_blocks_54m400k.pt"

train_blocks = tokenize_texts_to_blocks(
    tokenizer=trainer.tokenizer,
    texts=train_texts,
    block_size=trainer.model.config.block_size,
    cache_path=block_cache,
)

print(f"Token block tensor shape: {tuple(train_blocks.shape)}")

In [ ]:
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
elif DEVICE == "mps":
    torch.mps.empty_cache()

history = trainer.train(
    train_blocks,
    epochs=1,
    batch_size=8,
    learning_rate=3e-4,
    weight_decay=0.1,
    grad_clip=1.0,
    grad_accum_steps=4,
    cache_path=block_cache,
    num_workers=2,
    log_every=100,
)

output_dir = NOTEBOOK_DIR / "trained_model_54m400k_1epoch"
trainer.save(output_dir, epoch=1, global_step=len(history))
print(f"Saved checkpoint to: {output_dir}")

In [ ]:
sample_prompt = "Recipe for a cozy garlic mushroom pasta:\n\nIngredients:\n"
print(trainer.generate(sample_prompt, max_length=480, temperature=0.9, top_k=50, top_p=0.95, repetition_penalty=1.1))

## Load Trained Model from Checkpoint

In [ ]:
import importlib
importlib.reload(sys.modules['model2'])
from model2 import miniGPT

checkpoint_dir = NOTEBOOK_DIR / "trained_model_54m400k_1epoch"
loaded_trainer = miniGPT.load(str(checkpoint_dir), map_location=DEVICE)
loaded_trainer = loaded_trainer.to(DEVICE)
print(f"Model loaded from: {checkpoint_dir}")
print(f"Model is in eval mode: {not loaded_trainer.model.training}")

In [ ]:
prompts = [
    "Recipe for chocolate ",
    "Recipe for a delicious apple pie:\n\nIngredients:\n",
    "Ingredients: flour butter sugar",
    "Instructions: preheat oven",
    "How to make a",
    "Directions: mix until smooth",
]

print("=" * 70)
print("GENERATED RECIPES FROM LOADED MODEL (54M, 2 epochs)")
print("=" * 70)

for prompt in prompts:
    print(f"\nPrompt: {prompt}")
    print("-" * 70)
    generated = loaded_trainer.generate(
        prompt,
        max_length=300,
        temperature=0.85,
        top_k=50,
        top_p=0.95,
        repetition_penalty=1.2,
    )
    print(generated)
    print()